# 🤖 e-MİY Uygulaması — Otel Yorumlarından Müşteri Sesi Analizi

**Aksaray Üniversitesi — Yönetim Bilişim Sistemleri**  
**e-Müşteri İlişkileri Yönetimi Dersi**

---

Bu uygulamada şunları öğreneceğiz:
- Web'den müşteri yorumlarının otomatik toplanması (scraping)
- Ham metnin analize hazır hale getirilmesi (önişleme)
- Yorumların duygu analizi ile sınıflandırılması

**Nasıl çalışır?**  
Her bölümde bazı kod satırları **boş bırakılmıştır** — bunları siz dolduracaksınız.  
Boş yerler `# ✏️ BURAYA YAZ` ile işaretlidir.

---

## 📦 ADIM 0 — Kütüphaneleri Kur ve İçe Aktar

Aşağıdaki hücreyi **bir kez** çalıştır. Kurulum birkaç dakika sürebilir.

In [ ]:
# Kütüphane kurulumu (sadece ilk çalıştırmada gerekli)
import subprocess
subprocess.run(['pip', 'install', 'pandas', 'langdetect', 'nltk', 'textblob', '-q'])

print('✅ Kurulum tamamlandı!')

In [ ]:
# Kütüphaneleri içe aktar
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException
from textblob import TextBlob

# NLTK veri setlerini indir
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✅ Tüm kütüphaneler hazır!')

---
## 🌐 ADIM 1 — Veri Çekme (Web Scraping)

> **Not:** Web scraping için Selenium gerekir ve her bilgisayarda kurulum farklı davranabilir.  
> Bu yüzden veriler **önceden hazırlandı** — biz hazır CSV dosyasıyla çalışacağız.  
> Selenium kodunun nasıl çalıştığını derste birlikte inceleyeceğiz.

### Scraping Nedir?

Selenium, bir tarayıcıyı (Chrome, Firefox) **robot gibi kontrol eden** bir kütüphanedir.  
Tıklama, kaydırma, metin okuma — bunların hepsini Python ile yapabiliriz.

```python
# Böyle başlar:
driver = webdriver.Chrome()   # Tarayıcıyı aç
driver.get('https://...')     # Sayfaya git

# Yorumları bul:
yorumlar = driver.find_elements(By.CLASS_NAME, 'Review-comment-bodyText')
```

---

### 📂 Hazır veriyi yükle

In [ ]:
# CSV dosyasını oku
# (Dosyanın bu notebook ile aynı klasörde olduğundan emin ol)
df = pd.read_csv('agoda_yorumlar_hazir.csv')

# Kaç yorum var?
print(f'Toplam yorum sayısı: {len(df)}')
print()

# İlk 3 yoruma bakalım
print('--- İlk 3 yorum ---')
for i, yorum in enumerate(df['yorum'].head(3)):
    print(f'{i+1}. {yorum}')
    print()

In [ ]:
# 🔍 KEŞFET: DataFrame'in yapısını incele
# .shape → kaç satır, kaç sütun?
# .head() → ilk 5 satırı göster

print('DataFrame boyutu:', df.shape)
df.head()

---
## 🧹 ADIM 2 — Önişleme (Text Preprocessing)

Ham yorumları doğrudan analiz edemeyiz. Önce temizlememiz gerekiyor.

**Neden?**
- `"GREAT!!!"` ve `"great"` aynı şeydir → küçük harfe çevir
- `"!!!"` ve `","` analizi gürültülü yapar → noktalama kaldır  
- `"the"`, `"was"`, `"a"` anlam taşımaz → stopword çıkar

### 2.1 — Dil Tespiti

In [ ]:
# langdetect ile dil tespiti deneyelim
test_cumleleri = [
    "The hotel was absolutely fantastic!",
    "Otel çok güzeldi, kesinlikle tavsiye ederim.",
    "Das Hotel war wunderbar!",
    "素晴らしいホテルでした"
]

for cumle in test_cumleleri:
    dil = detect(cumle)
    print(f'{dil:5} → {cumle}')

In [ ]:
# Dil tespiti fonksiyonu
def ingilizce_mi(metin):
    """
    Metnin İngilizce olup olmadığını döndürür.
    True  → İngilizce
    False → Diğer diller
    """
    try:
        return detect(metin) == 'en'
    except LangDetectException:
        return False

# Tüm yorumlara uygula
df['ingilizce_mi'] = df['yorum'].apply(ingilizce_mi)

# Sonuç
print(f"Toplam yorum     : {len(df)}")
print(f"İngilizce yorum  : {df['ingilizce_mi'].sum()}")
print(f"Diğer dil        : {(~df['ingilizce_mi']).sum()}")

# Sadece İngilizce yorumları al
df_en = df[df['ingilizce_mi']].copy().reset_index(drop=True)
print(f"\nAnalize devam edecek yorum sayısı: {len(df_en)}")

### 2.2 — Metin Temizleme

Şimdi adım adım temizleme yapalım. Her adımı ayrı ayrı göreceğiz.

In [ ]:
# Örnek bir yorum seçelim
ornek = "The hotel was GREAT!!! Very, very good. 10/10 would recommend!!!"
print('HAM YORUM:')
print(ornek)
print()

# ADIM 1: Küçük harfe çevir
adim1 = ornek.lower()
print('ADIM 1 - Küçük harf:')
print(adim1)
print()

# ADIM 2: Noktalama ve özel karakterleri kaldır
# [^\w\s] → harf, rakam ve boşluk olmayan her şeyi sil
adim2 = re.sub(r'[^\w\s]', '', adim1)
print('ADIM 2 - Noktalama kaldır:')
print(adim2)
print()

# ADIM 3: Rakamları kaldır
adim3 = re.sub(r'\d+', '', adim2)
print('ADIM 3 - Rakamları kaldır:')
print(adim3)
print()

# ADIM 4: Tokenize (kelimelere ayır)
adim4 = nltk.word_tokenize(adim3)
print('ADIM 4 - Tokenize:')
print(adim4)
print()

# ADIM 5: Stopword'leri kaldır
STOPWORDS = set(stopwords.words('english'))
adim5 = [kelime for kelime in adim4 if kelime not in STOPWORDS]
print('ADIM 5 - Stopword kaldır:')
print(adim5)
print()
print('SONUÇ:', ' '.join(adim5))

In [ ]:
# ✏️ SENİN GÖREVIN — Temizleme fonksiyonunu tamamla
#
# Yukarıdaki 5 adımı bir araya getir ve fonksiyon olarak yaz.
# Boş yerleri doldur.

STOPWORDS = set(stopwords.words('english'))

def yorumu_temizle(yorum):
    """
    Ham yorum metnini temizler ve analize hazır hale getirir.
    """
    # ADIM 1: Küçük harfe çevir
    yorum = yorum.lower()
    
    # ADIM 2: Noktalama işaretlerini kaldır
    yorum = re.sub(r'[^\w\s]', '', yorum)
    
    # ADIM 3: Rakamları kaldır
    # ✏️ BURAYA YAZ
    
    # ADIM 4: Kelimelere ayır (tokenize)
    kelimeler = nltk.word_tokenize(yorum)
    
    # ADIM 5: Stopword'leri filtrele
    # ✏️ BURAYA YAZ
    
    kelimeler = ___
    


# Fonksiyonu test et
test = "The hotel was GREAT!!! Very, very good. 10/10 recommend!!!"
print('Girdi :', test)
print('Çıktı :', yorumu_temizle(test))

In [ ]:
# Fonksiyonu tüm yorumlara uygula
df_en['temiz_yorum'] = df_en['yorum'].apply(yorumu_temizle)

# Karşılaştır
# Not: df_en['sutun'][i] yerine .iloc[i] kullanıyoruz.
# Neden? Filtreleme sonrası index numaraları 0,1,2... olmayabilir;
# .iloc her zaman sıra numarasına göre güvenli erişir.
print('Önce ve sonra karşılaştırma:')
print('=' * 60)
for i in range(3):
    print(f'HAM    : {df_en["yorum"].iloc[i]}')
    print(f'TEMİZ  : {df_en["temiz_yorum"].iloc[i]}')
    print()

---
## 💬 ADIM 3 — Duygu Analizi (Sentiment Analysis)

TextBlob, bir cümleyi okuyup şu iki skoru verir:

| Skor | Aralık | Anlamı |
|------|--------|--------|
| **polarite** | -1.0 → +1.0 | Negatif → Pozitif |
| **öznellik** | 0.0 → 1.0 | Nesnel (olgusal) → Öznel (kişisel) |

### 3.1 — TextBlob'u Tanıyalım

In [ ]:
# TextBlob demosu — tahmin et, sonra çalıştır!
test_yorumlar = [
    "The hotel was absolutely fantastic and wonderful!",
    "Terrible experience. Dirty rooms and rude staff.",
    "The hotel is located near the train station.",
    "It was okay. Nothing special."
]

print(f"{'YORUM':55} {'POLARİTE':10} {'ÖZNELLİK':10}")
print('-' * 75)

for yorum in test_yorumlar:
    tb = TextBlob(yorum)
    pol = tb.sentiment.polarity
    ozn = tb.sentiment.subjectivity
    print(f"{yorum[:53]:55} {pol:+.2f}      {ozn:.2f}")

### 3.2 — Etiketleme

In [ ]:
# ✏️ SENİN GÖREVIN — Etiket fonksiyonunu tamamla
#
# Kural:
#   polarite > 0.1   → "Olumlu"
#   polarite < -0.1  → "Olumsuz"
#   arada            → "Nötr"

def duygu_etiketi(polarite):
    """
    Polarite skoruna göre duygu etiketi döndürür.
    """
    # ✏️ BURAYA YAZ — if/elif/else yapısı kur


# Test et
print(duygu_etiketi(0.8))   # → Olumlu
print(duygu_etiketi(-0.4))  # → Olumsuz
print(duygu_etiketi(0.05))  # → Nötr

In [ ]:
# Tüm yorumlara duygu analizi uygula
df_en['polarite']  = df_en['yorum'].apply(lambda x: TextBlob(x).sentiment.polarity)
df_en['oznellik']  = df_en['yorum'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
df_en['duygu']     = df_en['polarite'].apply(duygu_etiketi)

print('✅ Duygu analizi tamamlandı!')
print(f"\nAnaliz edilen yorum sayısı: {len(df_en)}")

---
## 📊 ADIM 4 — Sonuçları İncele

Artık elimizde temizlenmiş, etiketlenmiş bir veri seti var. Ne bulduk?

In [ ]:
# Duygu dağılımı
print('DUYGU DAĞILIMI')
print('=' * 30)
dagılım = df_en['duygu'].value_counts()
toplam = len(df_en)

for etiket, sayi in dagılım.items():
    yuzde = sayi / toplam * 100
    bar = '█' * int(yuzde / 3)
    print(f"{etiket:10} {sayi:3} yorum  ({yuzde:.0f}%)  {bar}")

print()
print(f"Ortalama polarite : {df_en['polarite'].mean():.3f}")
print(f"Ortalama öznellik : {df_en['oznellik'].mean():.3f}")

In [ ]:
# En olumlu 3 yorum
print('🌟 EN OLUMLU 3 YORUM')
print('=' * 60)
en_olumlu = df_en.nlargest(3, 'polarite')[['yorum', 'polarite']]
for _, satir in en_olumlu.iterrows():
    print(f"  Skor: {satir['polarite']:+.2f}")
    print(f"  {satir['yorum']}")
    print()

# En olumsuz 3 yorum
print('⚠️  EN OLUMSUZ 3 YORUM')
print('=' * 60)
en_olumsuz = df_en.nsmallest(3, 'polarite')[['yorum', 'polarite']]
for _, satir in en_olumsuz.iterrows():
    print(f"  Skor: {satir['polarite']:+.2f}")
    print(f"  {satir['yorum']}")
    print()

In [ ]:
# ✏️ SENİN GÖREVIN — Sadece olumsuz yorumları filtrele
#
# İpucu: df_en[df_en['duygu'] == '...'] şeklinde filtrele

olumsuz_yorumlar = # ✏️ BURAYA YAZ

print(f"Olumsuz yorum sayısı: {len(olumsuz_yorumlar)}")
print()
print('İlk 3 olumsuz yorum:')
for yorum in olumsuz_yorumlar['yorum'].head(3):
    print(f'  → {yorum}')

---
## 💾 ADIM 5 — Sonuçları Kaydet

In [ ]:
# Sonuçları CSV'ye yaz
cikti = df_en[['yorum', 'temiz_yorum', 'polarite', 'oznellik', 'duygu']]
cikti.to_csv('sonuclar.csv', index=False, encoding='utf-8-sig')

print('✅ sonuclar.csv dosyası oluşturuldu!')
print(f'   {len(cikti)} satır, {len(cikti.columns)} sütun')
print()
print('Sütunlar:', list(cikti.columns))

---
## 🎯 BONUS — e-MİY Bağlantısı: Bu Analiz Bize Ne Söylüyor?

Aşağıdaki soruları düşün ve markdown hücresi ekleyerek cevapla:

In [ ]:
# BONUS SORU 1
# Olumsuz yorumlarda hangi kelimeler sık geçiyor?
# İpucu: temiz_yorum sütununu kullan, kelimeleri say

from collections import Counter

# Sadece olumsuz yorumları al
olumsuz = df_en[df_en['duygu'] == 'Olumsuz']['temiz_yorum']

# Tüm kelimeleri birleştir ve say
tum_kelimeler = ' '.join(olumsuz).split()
en_sik = Counter(tum_kelimeler).most_common(10)

print('Olumsuz yorumlarda en sık geçen kelimeler:')
for kelime, sayi in en_sik:
    bar = '▪' * sayi
    print(f'  {kelime:15} {sayi:3}  {bar}')

In [ ]:
# BONUS SORU 2
# ✏️ SENİN GÖREVIN
#
# Olumlu yorumlarda en sık geçen 10 kelimeyi bul.
# Yukarıdaki kodu model alarak yaz.

# ✏️ BURAYA YAZ

---

## 🏁 Tebrikler!

Bu uygulamada şunları başardın:

✅ Ham metin verisini pandas DataFrame'e yükleme  
✅ Dil tespiti ile ilgili yorumları filtreleme  
✅ Regex ve NLTK ile metin temizleme  
✅ TextBlob ile duygu analizi yapma  
✅ Sonuçları CSV dosyasına kaydetme  

